<a href="https://colab.research.google.com/github/MizushimaToshihiko/Kaggle/blob/main/ps-s6e4-Predicting-Irrigation-Need/experiments/exp_20260416_047a_lgb025_optuna_resume/PS_S6E4_LGB025_Optuna_Only.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# IMPORTANT: SOME KAGGLE DATA SOURCES ARE PRIVATE
# RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES.
import kagglehub
kagglehub.login()


Kaggle credentials set.
Kaggle credentials successfully validated.


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.

playground_series_s6e4_path = kagglehub.competition_download('playground-series-s6e4')
miadul_irrigation_water_requirement_prediction_dataset_path = kagglehub.dataset_download('miadul/irrigation-water-requirement-prediction-dataset')

print('Data source import complete.')


100%|██████████| 32.5M/32.5M [00:00<00:00, 120MB/s]

Extracting files...


100%|██████████| 342k/342k [00:00<00:00, 100MB/s]

Extracting files...
Data source import complete.


In [5]:
# =========================================================
# PS S6E4 - exp_20260416_047a_lgb025_optuna_resume
# Optuna only for 025 LGB digit TE model
# resumable at trial level via SQLite storage
# =========================================================

In [6]:
!pip install Optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.5/419.5 kB 12.6 MB/s eta 0:00:00


In [7]:
import os
import gc
import json
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import optuna
import lightgbm as lgb
import torch

from sklearn.model_selection import KFold
from sklearn.metrics import balanced_accuracy_score

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 300)
pd.set_option("display.width", 200)

In [8]:
# ---------------------------------------------------------
# CFG
# ---------------------------------------------------------
class CFG:
    EXP_ID = "exp_20260416_047a_lgb025_optuna_resume"
    EXP_NAME = "lgb025_optuna_resume"

    SAVE_DIR = Path(f"/content/drive/MyDrive/Kaggle/PS S6E4 Predicting Irrigation Need/{EXP_ID}")
    SAVE_DIR.mkdir(parents=True, exist_ok=True)

    TRAIN_PATH = "/content/drive/MyDrive/Kaggle/PS S6E4 Predicting Irrigation Need/train.csv"
    TEST_PATH = "/content/drive/MyDrive/Kaggle/PS S6E4 Predicting Irrigation Need/test.csv"

    TARGET = "Irrigation_Need"
    ID_COL = "id"

    SEED = 2026
    N_SPLITS = 5
    NUM_CLASSES = 3

    N_TRIALS_TOTAL = 10   # 途中停止後に再開する前提の総目標 trial 数
    STUDY_NAME = "ps_s6e4_047a_lgb025_optuna_resume"
    STORAGE_PATH = SAVE_DIR / "optuna_study.db"
    STORAGE_URL = f"sqlite:///{STORAGE_PATH}"

    # prune を入れて重い bad trial を早めに切る
    PRUNER_N_STARTUP_TRIALS = 1
    PRUNER_N_WARMUP_STEPS = 1

    # まずは軽め
    EARLY_STOPPING_ROUNDS = 100

In [9]:
# ---------------------------------------------------------
# Utils
# ---------------------------------------------------------
def save_json(obj, path: str):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)

def seed_everything(seed: int):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)

def balanced_acc_score_mc(y_true, y_pred):
    if len(y_pred.shape) == 2:
        y_pred = np.argmax(y_pred, axis=1)
    c = len(np.unique(y_true))
    acc = 0.0
    for i in range(c):
        acc += np.sum((y_true == i) & (y_pred == i)) / np.sum(y_true == i) / c
    return acc


seed_everything(CFG.SEED)

# ---------------------------------------------------------
# Load data
# ---------------------------------------------------------
train = pd.read_csv(CFG.TRAIN_PATH)
test = pd.read_csv(CFG.TEST_PATH)

print(f"train.shape={train.shape}, test.shape={test.shape}")

target2idx = {v: i for i, v in enumerate(train[CFG.TARGET].unique())}
idx2target = {v: k for k, v in target2idx.items()}

train[CFG.TARGET] = train[CFG.TARGET].map(target2idx)
y = train[CFG.TARGET].values

print(train.head())

train.shape=(630000, 21), test.shape=(270000, 20)
   id Soil_Type  Soil_pH  Soil_Moisture  Organic_Carbon  Electrical_Conductivity  Temperature_C  Humidity  Rainfall_mm  Sunlight_Hours  Wind_Speed_kmh  Crop_Type Crop_Growth_Stage  Season  \
0   0     Loamy     4.92          32.58            1.01                     3.05          15.01     50.61       725.99            5.90           16.79  Sugarcane            Sowing    Zaid   
1   1      Clay     7.08          56.61            0.44                     2.00          22.92     67.86       985.66            6.98            3.39      Wheat        Vegetative  Kharif   
2   2      Clay     5.69          27.71            0.81                     2.83          26.97     92.22      2201.70            6.05            3.85       Rice        Vegetative  Kharif   
3   3     Sandy     5.65          13.32            1.33                     0.87          13.32     61.57      1357.33            9.12            2.31      Wheat         Flowering  Khari

In [10]:
class OrderedTE:
    def __init__(self, a=1):
        self.a = a

    def fit(self, train, category_cols=None, target_col="target"):
        if category_cols is None:
            category_cols = []

        self.train = train.copy()
        self.TARGET = target_col
        self.category_cols = category_cols

        self.classes_ = sorted(train[target_col].unique())
        self.num_classes_ = len(self.classes_)
        self.global_prior_ = train[target_col].value_counts(normalize=True).sort_index().values

        for c in self.category_cols:
            stats_list = []

            for k, cls in enumerate(self.classes_):
                y_binary = (train[target_col] == cls).astype(int)

                df = train[[c]].copy()
                df["y"] = y_binary.values
                df["cnt"] = 1

                df["cum_cnt"] = df.groupby(c)["cnt"].cumsum() - df["cnt"]
                df["cum_sum"] = df.groupby(c)["y"].cumsum() - df["y"]

                smooth_prior = self.a * self.global_prior_[k]

                te_col = f"{c}_TE_cls{cls}"
                df[te_col] = (df["cum_sum"] + smooth_prior) / (df["cum_cnt"] + self.a)
                df.loc[df["cum_cnt"] == 0, te_col] = self.global_prior_[k]

                self.train[te_col] = df[te_col].values

                stats_df = df.groupby(c)["y"].agg(["count", "sum"]).reset_index()
                stats_df.columns = [c, f"{c}_count_cls{cls}", f"{c}_sum_cls{cls}"]
                stats_df[f"{c}_prior_cls{cls}"] = self.global_prior_[k]
                stats_list.append(stats_df)

            combined_stats = stats_list[0]
            for i in range(1, len(stats_list)):
                combined_stats = combined_stats.merge(stats_list[i], on=c, how="outer")
            setattr(self, f"{c}_stats", combined_stats)

        return self.train

    def transform(self, test):
        test = test.copy()

        for c in self.category_cols:
            stats_df = getattr(self, f"{c}_stats")
            test = test.merge(stats_df, on=c, how="left")

            for k, cls in enumerate(self.classes_):
                te_col = f"{c}_TE_cls{cls}"
                count_col = f"{c}_count_cls{cls}"
                sum_col = f"{c}_sum_cls{cls}"
                prior_col = f"{c}_prior_cls{cls}"

                if count_col in test.columns:
                    test[te_col] = (test[sum_col] + self.a * test[prior_col]) / (test[count_col] + self.a)
                    test[te_col] = test[te_col].fillna(test[prior_col])
                    test.drop([count_col, sum_col, prior_col], axis=1, inplace=True)
                else:
                    test[te_col] = self.global_prior_[k]

        return test

In [11]:
def build_features_025(train_df: pd.DataFrame, test_df: pd.DataFrame):
    """
    025 の feature pipeline のうち、
    - digit features
    - constant-column drop
    - frequency-based category remap
    までを作る。

    OrderedTE は fold 内でかけるので、ここではまだ実行しない。

    Returns
    -------
    X_train_base : pd.DataFrame
        target を除いた base features（TE 前）
    X_test_base : pd.DataFrame
        test 側の base features（TE 前）
    meta : dict
        fold 内 TE で使うメタ情報
    """
    train_df = train_df.copy()
    test_df = test_df.copy()

    target_col = CFG.TARGET
    id_col = CFG.ID_COL

    # id は落とす
    if id_col in train_df.columns:
        train_df = train_df.drop(columns=[id_col])
    if id_col in test_df.columns:
        test_df = test_df.drop(columns=[id_col])

    # target は train にだけある
    CATS = [c for c in test_df.columns if train_df[c].dtype == object]
    NUMS = [c for c in test_df.columns if c not in CATS]

    M = train_df[NUMS].max()

    def add_digit_features(df):
        df = df.copy()
        for c in NUMS:
            for k in range(-4, 4):
                df[f"{c}_digit{k}"] = (df[c] // (10 ** k) % 10).astype("int8")

            if M[c] < 10:
                df[c] = df[c].round(3)
            elif M[c] < 100:
                df[c] = df[c].round(2)
            else:
                df[c] = df[c].round(1)
        return df

    train_df = add_digit_features(train_df)
    test_df = add_digit_features(test_df)

    # 定数列 drop は train/test 両方を見て安全に決める
    base_test_cols = [c for c in test_df.columns if c != target_col]
    DROP = [c for c in base_test_cols if test_df[c].nunique() == 1]
    if DROP:
        train_df = train_df.drop(columns=DROP, errors="ignore")
        test_df = test_df.drop(columns=DROP, errors="ignore")

    # digit 列をカテゴリ扱い
    CATEGORY = CATS + [c for c in test_df.columns if "digit" in c]
    FEATURES = CATEGORY + NUMS

    # frequency-based remap
    for c in CATEGORY:
        freq = train_df[c].value_counts()
        mapping = {val: idx for idx, (val, count) in enumerate(freq[freq >= 5].items())}
        mapping_default = len(mapping)

        train_df[c] = train_df[c].map(lambda x: mapping.get(x, mapping_default))
        test_df[c] = test_df[c].map(lambda x: mapping.get(x, mapping_default))

    # base features only
    X_train_base = train_df[FEATURES].copy()
    X_test_base = test_df[FEATURES].copy()

    meta = {
        "CATS": CATS,
        "NUMS": NUMS,
        "CATEGORY": CATEGORY,
        "FEATURES": FEATURES,
        "DROP": DROP,
        "target_col": target_col,
    }

    return X_train_base, X_test_base, meta


X_train_base, X_test_base, feature_meta = build_features_025(train.copy(), test.copy())

print("X_train.shape =", X_train_base.shape)
print("X_test.shape  =", X_test_base.shape)
for k, v in feature_meta .items():
    if k is not "target_col": print(f"{k} :", len(v))
    else : print(f"{k} : {v}")

X_train.shape = (630000, 85)
X_test.shape  = (270000, 85)
CATS : 8
NUMS : 11
CATEGORY : 74
FEATURES : 85
DROP : 22
target_col : Irrigation_Need


In [12]:
# ---------------------------------------------------------
# Optuna search space
# ---------------------------------------------------------
def make_lgb_params(trial: optuna.trial.Trial):
    return {
        "objective": "multiclass",
        "num_class": CFG.NUM_CLASSES,
        "metric": "multi_logloss",
        "verbosity": -1,
        "boosting_type": "gbdt",
        "seed": CFG.SEED,

        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.10, log=True),
        "num_leaves": trial.suggest_int("num_leaves", 16, 128, step=8),
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "min_data_in_leaf": trial.suggest_int("min_data_in_leaf", 20, 200, step=10),
        "feature_fraction": trial.suggest_float("feature_fraction", 0.60, 1.0),
        "bagging_fraction": trial.suggest_float("bagging_fraction", 0.60, 1.0),
        "bagging_freq": trial.suggest_int("bagging_freq", 1, 10),
        "lambda_l1": trial.suggest_float("lambda_l1", 1e-8, 10.0, log=True),
        "lambda_l2": trial.suggest_float("lambda_l2", 1e-8, 20.0, log=True),
        "min_gain_to_split": trial.suggest_float("min_gain_to_split", 0.0, 2.0),
        "max_bin": trial.suggest_int("max_bin", 127, 511, step=64),

        # まずは軽め
        "n_estimators": trial.suggest_int("n_estimators", 500, 2000, step=250),

        # Kaggle GPU が使えるなら使う
        "device_type": "gpu" if torch.cuda.is_available() else "cpu",
    }


def cv_score_lgb(params, X_base, y, X_test_base, feature_meta, trial=None):
    kf = KFold(n_splits=CFG.N_SPLITS, shuffle=True, random_state=42)
    oof = np.zeros((len(X_base), 3), dtype=np.float32)
    fold_scores = []

    CATEGORY = feature_meta["CATEGORY"]
    CATS = feature_meta["CATS"]
    target_col = CFG.TARGET

    for fold, (tr_idx, va_idx) in enumerate(kf.split(X_base, y), 1):
        X_tr_base = X_base.iloc[tr_idx].copy()
        X_va_base = X_base.iloc[va_idx].copy()

        y_tr = pd.Series(y[tr_idx], name=target_col)
        y_va = y[va_idx]

        unique, counts = np.unique(y_tr.values, return_counts=True)
        count_dict = dict(zip(unique, counts))
        avg_count = len(y_tr) / len(unique)
        weights_dict = {cls: avg_count / cnt for cls, cnt in count_dict.items()}
        train_weights = np.array([weights_dict[v] for v in y_tr.values], dtype=np.float32)

        te = OrderedTE(a=1)

        full_df = pd.concat([X_tr_base.reset_index(drop=True), y_tr.reset_index(drop=True)], axis=1)
        full_df["weight"] = train_weights

        te_train = te.fit(
            full_df.sample(frac=1, random_state=42),
            category_cols=CATEGORY,
            target_col=target_col,
        )

        X_tr = te_train.drop([target_col, "weight"], axis=1)
        y_tr_fit = te_train[target_col].values
        train_weights_fit = te_train["weight"].values

        X_va = te.transform(X_va_base.reset_index(drop=True))

        X_tr = X_tr.drop(columns=CATS, errors="ignore")
        X_va = X_va.drop(columns=CATS, errors="ignore")

        model = lgb.LGBMClassifier(**params)

        model.fit(
            X_tr,
            y_tr_fit,
            eval_set=[(X_va, y_va)],
            eval_metric="multi_logloss",
            callbacks=[
                lgb.early_stopping(CFG.EARLY_STOPPING_ROUNDS, verbose=False),
                lgb.log_evaluation(0),
            ],
        )

        va_pred = model.predict_proba(X_va)
        oof[va_idx] = va_pred

        sc = float(balanced_accuracy_score(y_va, np.argmax(va_pred, axis=1)))
        fold_scores.append(sc)
        print(f"fold {fold} BA: {sc:.6f}")

        if trial is not None:
            trial.report(np.mean(fold_scores), step=fold)
            if trial.should_prune():
                raise optuna.TrialPruned()

        del model, X_tr, X_va, y_tr_fit, train_weights_fit, va_pred
        gc.collect()

    full_score = float(balanced_accuracy_score(y, np.argmax(oof, axis=1)))
    return full_score, fold_scores, oof


def objective(trial: optuna.trial.Trial):
    params = make_lgb_params(trial)
    score, fold_scores, _ = cv_score_lgb(params, X_train_base, y, X_test_base, feature_meta, trial=trial)
    trial.set_user_attr("fold_scores", fold_scores)
    return score

# ---------------------------------------------------------
# Save callback after each completed trial
# ---------------------------------------------------------
def save_progress(study: optuna.Study, trial: optuna.trial.FrozenTrial):
    trials_df = study.trials_dataframe()
    trials_df.to_csv(CFG.SAVE_DIR / "optuna_trials.csv", index=False)

    payload = {
        "exp_id": CFG.EXP_ID,
        "base_parent": "exp_20260409_025_lgb_digit_te_min",
        "study_name": CFG.STUDY_NAME,
        "best_trial_number": int(study.best_trial.number),
        "best_cv_raw": float(study.best_value),
        "best_fold_scores": study.best_trial.user_attrs.get("fold_scores", None),
        "best_params": study.best_trial.params,
        "n_trials_completed": len([t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE]),
        "n_trials_pruned": len([t for t in study.trials if t.state == optuna.trial.TrialState.PRUNED]),
        "n_trials_total_seen": len(study.trials),
        "notes": {
            "phase": "optuna_only_resume",
            "retrain_not_done_yet": True,
            "bias_not_applied_yet": True,
            "resume_unit": "trial",
        },
    }
    save_json(payload, CFG.SAVE_DIR / "best_params.json")
    save_json(payload, CFG.SAVE_DIR / "cv_result.json")

In [ ]:
# ---------------------------------------------------------
# Study: resumable at trial level
# ---------------------------------------------------------
study = optuna.create_study(
    study_name=CFG.STUDY_NAME,
    storage=CFG.STORAGE_URL,
    load_if_exists=True,
    direction="maximize",
    pruner=optuna.pruners.MedianPruner(
        n_startup_trials=CFG.PRUNER_N_STARTUP_TRIALS,
        n_warmup_steps=CFG.PRUNER_N_WARMUP_STEPS,
    ),
)

n_complete_before = len([t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE])
print("complete trials before resume:", n_complete_before)
print("storage:", CFG.STORAGE_URL)

remaining_trials = max(CFG.N_TRIALS_TOTAL - n_complete_before, 0)
print("remaining_trials to run now:", remaining_trials)

if remaining_trials > 0:
    study.optimize(
        objective,
        n_trials=remaining_trials,
        callbacks=[save_progress],
        gc_after_trial=True,
        show_progress_bar=True,
    )
else:
    print("Target number of completed trials already reached.")

# optimize 後に最後の保存
save_progress(study, study.best_trial)

print("\nbest_value:", study.best_value)
print("best_params:", study.best_params)
print("best_trial_number:", study.best_trial.number)
print("saved to:", CFG.SAVE_DIR)

[I 2026-04-17 04:50:20,840] A new study created in RDB with name: ps_s6e4_047a_lgb025_optuna_resume


complete trials before resume: 0
storage: sqlite:////content/drive/MyDrive/Kaggle/PS S6E4 Predicting Irrigation Need/exp_20260416_047a_lgb025_optuna_resume/optuna_study.db
remaining_trials to run now: 10


  0%|          | 0/10 [00:00<?, ?it/s]

fold 1 BA: 0.962055
fold 2 BA: 0.963462
fold 3 BA: 0.961582
fold 4 BA: 0.966698
fold 5 BA: 0.962564
[I 2026-04-17 04:58:29,336] Trial 0 finished with value: 0.963266969575321 and parameters: {'learning_rate': 0.06132295247446743, 'num_leaves': 96, 'max_depth': 6, 'min_data_in_leaf': 180, 'feature_fraction': 0.6466492708839899, 'bagging_fraction': 0.8067160331257692, 'bagging_freq': 7, 'lambda_l1': 0.002506183042169745, 'lambda_l2': 0.0024550725978754843, 'min_gain_to_split': 1.2447777757807341, 'max_bin': 127, 'n_estimators': 1000}. Best is trial 0 with value: 0.963266969575321.
